In [0]:
spark.sql("SHOW TABLES IN workspace.google_fhir_data").show()

+----------------+--------------------+-----------+
|        database|           tableName|isTemporary|
+----------------+--------------------+-----------+
|google_fhir_data|  allergyintolerance|      false|
|google_fhir_data|           claim_001|      false|
|google_fhir_data|           claim_002|      false|
|google_fhir_data|           claim_003|      false|
|google_fhir_data|           condition|      false|
|google_fhir_data|diagnosticreport_001|      false|
|google_fhir_data|diagnosticreport_002|      false|
|google_fhir_data|diagnosticreport_003|      false|
|google_fhir_data|       encounter_001|      false|
|google_fhir_data|       encounter_002|      false|
|google_fhir_data|   medicationrequest|      false|
|google_fhir_data|     observation_001|      false|
|google_fhir_data|     observation_002|      false|
|google_fhir_data|     observation_003|      false|
|google_fhir_data|     observation_004|      false|
|google_fhir_data|     observation_005|      false|
|google_fhir

In [0]:
from pyspark.sql.functions import col, get_json_object

patients = spark.table("workspace.google_fhir_data.patient").select(
    col("id").alias("patient_id"),
    get_json_object(col("resource"), "$.name[0].family").alias("last_name"),
    get_json_object(col("resource"), "$.name[0].given[0]").alias("first_name"),
    get_json_object(col("resource"), "$.gender").alias("gender"),
    get_json_object(col("resource"), "$.birthDate").alias("birth_date"),
    get_json_object(col("resource"), "$.address[0].city").alias("city"),
    get_json_object(col("resource"), "$.address[0].state").alias("state"),
    get_json_object(col("resource"), "$.deceasedDateTime").alias("deceased_date")
)

patients.show(5, truncate=False)

+------------------------------------+----------+----------+------+----------+-----------+-----+-------------------------+
|patient_id                          |last_name |first_name|gender|birth_date|city       |state|deceased_date            |
+------------------------------------+----------+----------+------+----------+-----------+-----+-------------------------+
|1a4f7487-d2e6-408c-44a8-ca456c6e38bb|Jacobs452 |Octavia629|female|1963-01-17|San Antonio|TX   |2007-11-22T00:10:46-06:00|
|7d263602-8092-d0dc-3a46-995f0d6ddb33|Reilly981 |Donnell534|male  |1977-06-24|San Antonio|TX   |NULL                     |
|5465468a-cf0a-4006-3691-1525cfcd2da7|Boehm581  |Hai304    |male  |1981-01-27|San Antonio|TX   |NULL                     |
|91592d7b-43f9-b288-0255-e44789759a0a|Sanford861|Alexia646 |female|2014-10-10|San Antonio|TX   |NULL                     |
|7ad43115-b04e-9960-e5b4-0b82031b610f|Kemmer137 |Wilburn655|male  |2014-06-18|San Antonio|TX   |NULL                     |
+---------------

In [0]:
from pyspark.sql.functions import col, get_json_object, unix_timestamp, datediff

encounters = spark.table("workspace.google_fhir_data.encounter_001").union(
    spark.table("workspace.google_fhir_data.encounter_002")
).select(
    col("id").alias("encounter_id"),
    get_json_object(col("resource"), "$.subject.reference").alias("patient_ref"),
    get_json_object(col("resource"), "$.status").alias("status"),
    get_json_object(col("resource"), "$.class.code").alias("encounter_class"),
    get_json_object(col("resource"), "$.type[0].coding[0].display").alias("encounter_type"),
    get_json_object(col("resource"), "$.period.start").alias("start_date"),
    get_json_object(col("resource"), "$.period.end").alias("end_date")
)

encounters.show(5, truncate=False)

+------------------------------------+---------------------------------------------+--------+---------------+------------------------------------------+-------------------------+-------------------------+
|encounter_id                        |patient_ref                                  |status  |encounter_class|encounter_type                            |start_date               |end_date                 |
+------------------------------------+---------------------------------------------+--------+---------------+------------------------------------------+-------------------------+-------------------------+
|10ca2e4f-9402-1da8-7aed-17632efc91b4|urn:uuid:10ca2e4f-9402-1da8-54f1-059e62503949|finished|AMB            |Consultation for treatment (procedure)    |2016-01-01T20:17:39-06:00|2016-01-01T20:32:39-06:00|
|57515f29-7602-dcbf-c9d2-7c1cc20cee8a|urn:uuid:57515f29-7602-dcbf-0ab3-b279c0903d36|finished|AMB            |Encounter for problem (procedure)         |2008-10-24T12:03:55-05:00|20

In [0]:
from pyspark.sql.functions import col, get_json_object, regexp_replace, to_timestamp

encounters_clean = (
    spark.table("workspace.google_fhir_data.encounter_001")
    .union(spark.table("workspace.google_fhir_data.encounter_002"))
    .select(
        col("id").alias("encounter_id"),
        get_json_object(col("resource"), "$.subject.reference").alias("patient_ref"),
        get_json_object(col("resource"), "$.status").alias("status"),
        get_json_object(col("resource"), "$.class.code").alias("encounter_class"),
        get_json_object(col("resource"), "$.type[0].coding[0].display").alias("encounter_type"),
        get_json_object(col("resource"), "$.period.start").alias("start_date"),
        get_json_object(col("resource"), "$.period.end").alias("end_date")
    )
    .withColumn("patient_id", regexp_replace(col("patient_ref"), "urn:uuid:", ""))
    .withColumn("start_ts", to_timestamp(col("start_date")))
    .withColumn("end_ts", to_timestamp(col("end_date")))
    .drop("patient_ref")
)

encounters_clean.filter(col("encounter_class") == "IMP").show(5, truncate=False)

+------------------------------------+--------+---------------+-------------------------------------------------------+-------------------------+-------------------------+------------------------------------+-------------------+-------------------+
|encounter_id                        |status  |encounter_class|encounter_type                                         |start_date               |end_date                 |patient_id                          |start_ts           |end_ts             |
+------------------------------------+--------+---------------+-------------------------------------------------------+-------------------------+-------------------------+------------------------------------+-------------------+-------------------+
|307ab11f-ff8e-63d6-491f-7e858484a38a|finished|IMP            |Hospital admission (procedure)                         |2017-06-13T22:07:26-05:00|2017-07-16T22:22:26-05:00|307ab11f-ff8e-63d6-fb00-b97e91b2234e|2017-06-14 03:07:26|2017-07-17 03:22:26|
|eed

In [0]:
encounters_clean.filter(col("encounter_class") == "IMP").count()

116

In [0]:
from pyspark.sql.functions import datediff
from pyspark.sql import Window
import pyspark.sql.functions as F

# Get only inpatient encounters
inpatient = encounters_clean.filter(col("encounter_class") == "IMP")

# Self join to find readmissions
readmissions = inpatient.alias("a").join(
    inpatient.alias("b"),
    on=[
        col("a.patient_id") == col("b.patient_id"),
        col("b.start_ts") > col("a.end_ts"),
        datediff(col("b.start_ts"), col("a.end_ts")) <= 30
    ]
).select(
    col("a.patient_id"),
    col("a.encounter_id").alias("index_encounter_id"),
    col("a.start_ts").alias("index_admission"),
    col("a.end_ts").alias("index_discharge"),
    col("b.encounter_id").alias("readmission_encounter_id"),
    col("b.start_ts").alias("readmission_date"),
    datediff(col("b.start_ts"), col("a.end_ts")).alias("days_to_readmission")
)

readmissions.show(10, truncate=False)

+------------------------------------+------------------------------------+-------------------+-------------------+------------------------------------+-------------------+-------------------+
|patient_id                          |index_encounter_id                  |index_admission    |index_discharge    |readmission_encounter_id            |readmission_date   |days_to_readmission|
+------------------------------------+------------------------------------+-------------------+-------------------+------------------------------------+-------------------+-------------------+
|4a95bdc7-c1f1-8a09-509a-cf6bef54fc75|4a95bdc7-c1f1-8a09-563a-e5f0503ff579|2023-01-30 20:12:03|2023-01-31 20:12:03|4a95bdc7-c1f1-8a09-89e4-1693a1944ea8|2023-02-02 04:39:35|2                  |
|9589dfd6-09ea-113d-ca42-4f09df70f4a9|9589dfd6-09ea-113d-0f09-f57f5d2c3fea|2024-01-12 16:09:10|2024-02-07 00:12:10|9589dfd6-09ea-113d-3999-851ba5b5bb36|2024-02-10 16:09:10|3                  |
|006232dd-9560-5c9a-1430-c5ceaacb45

In [0]:
from pyspark.sql.functions import when

# Get list of index encounters that led to readmission
readmitted_encounters = readmissions.select(
    col("index_encounter_id")
).distinct()

# Label all inpatient encounters using aliases
labeled_encounters = (
    inpatient.alias("i")
    .join(
        readmitted_encounters.alias("r"),
        col("i.encounter_id") == col("r.index_encounter_id"),
        how="left"
    )
    .withColumn(
        "readmitted_30day",
        when(col("r.index_encounter_id").isNotNull(), 1).otherwise(0)
    )
    .select(
        col("i.encounter_id"),
        col("i.patient_id"),
        col("i.encounter_type"),
        col("i.start_ts"),
        col("i.end_ts"),
        col("readmitted_30day")
    )
)

labeled_encounters.groupBy("readmitted_30day").count().show()

+----------------+-----+
|readmitted_30day|count|
+----------------+-----+
|               1|    4|
|               0|  112|
+----------------+-----+



In [0]:
from pyspark.sql.functions import datediff, current_date, floor

patients_clean = spark.table("workspace.google_fhir_data.patient").select(
    col("id").alias("patient_id"),
    get_json_object(col("resource"), "$.gender").alias("gender"),
    get_json_object(col("resource"), "$.birthDate").alias("birth_date"),
    get_json_object(col("resource"), "$.deceasedDateTime").alias("deceased_date")
)

# Join encounters with patients
encounters_with_demographics = labeled_encounters.alias("e").join(
    patients_clean.alias("p"),
    col("e.patient_id") == col("p.patient_id"),
    how="left"
).withColumn(
    "age_at_admission",
    floor(datediff(col("e.start_ts"), to_timestamp(col("p.birth_date"))) / 365.25)
).select(
    col("e.encounter_id"),
    col("e.patient_id"),
    col("e.encounter_type"),
    col("e.start_ts"),
    col("e.end_ts"),
    col("e.readmitted_30day"),
    col("p.gender"),
    col("age_at_admission")
)

encounters_with_demographics.show(5, truncate=False)

+------------------------------------+------------------------------------+-------------------------------------------------------+-------------------+-------------------+----------------+------+----------------+
|encounter_id                        |patient_id                          |encounter_type                                         |start_ts           |end_ts             |readmitted_30day|gender|age_at_admission|
+------------------------------------+------------------------------------+-------------------------------------------------------+-------------------+-------------------+----------------+------+----------------+
|307ab11f-ff8e-63d6-491f-7e858484a38a|307ab11f-ff8e-63d6-fb00-b97e91b2234e|Hospital admission (procedure)                         |2017-06-14 03:07:26|2017-07-17 03:22:26|0               |male  |65              |
|eed8a921-eac8-0778-5f1a-0f28b8c2a227|eed8a921-eac8-0778-4113-255f4e35506a|Drug rehabilitation and detoxification (regime/therapy)|2016-08-05 06:14:

In [0]:
from pyspark.sql.functions import round as spark_round

encounters_featured = encounters_with_demographics.withColumn(
    "length_of_stay_days",
    spark_round(
        (col("end_ts").cast("long") - col("start_ts").cast("long")) / 86400, 1
    )
)

encounters_featured.select(
    "encounter_id", "gender", "age_at_admission", 
    "length_of_stay_days", "readmitted_30day"
).show(5, truncate=False)

+------------------------------------+------+----------------+-------------------+----------------+
|encounter_id                        |gender|age_at_admission|length_of_stay_days|readmitted_30day|
+------------------------------------+------+----------------+-------------------+----------------+
|307ab11f-ff8e-63d6-491f-7e858484a38a|male  |65              |33.0               |0               |
|eed8a921-eac8-0778-5f1a-0f28b8c2a227|female|40              |4.0                |0               |
|0afc0e1f-71ef-2925-ef91-adc6333145ba|male  |39              |5.0                |0               |
|72b5373b-e9f1-db2d-30db-67fa208b7359|male  |25              |1.0                |0               |
|47697c8b-11d6-e042-4e39-c3b590e548b7|female|74              |1.0                |0               |
+------------------------------------+------+----------------+-------------------+----------------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import count

# Parse conditions
conditions = spark.table("workspace.google_fhir_data.condition").select(
    get_json_object(col("resource"), "$.subject.reference").alias("patient_ref"),
    get_json_object(col("resource"), "$.code.coding[0].display").alias("condition_name"),
    get_json_object(col("resource"), "$.clinicalStatus.coding[0].code").alias("clinical_status")
).withColumn(
    "patient_id", regexp_replace(col("patient_ref"), "urn:uuid:", "")
).drop("patient_ref")

# Count active conditions per patient
condition_counts = conditions.filter(
    col("clinical_status") == "active"
).groupBy("patient_id").agg(
    count("condition_name").alias("active_condition_count")
)

# Join to our feature set
encounters_featured = encounters_featured.alias("e").join(
    condition_counts.alias("c"),
    col("e.patient_id") == col("c.patient_id"),
    how="left"
).select(
    col("e.*"),
    col("c.active_condition_count")
).fillna(0, subset=["active_condition_count"])

encounters_featured.select(
    "encounter_id", "gender", "age_at_admission",
    "length_of_stay_days", "active_condition_count", "readmitted_30day"
).show(5, truncate=False)

+------------------------------------+------+----------------+-------------------+----------------------+----------------+
|encounter_id                        |gender|age_at_admission|length_of_stay_days|active_condition_count|readmitted_30day|
+------------------------------------+------+----------------+-------------------+----------------------+----------------+
|307ab11f-ff8e-63d6-491f-7e858484a38a|male  |65              |33.0               |13                    |0               |
|eed8a921-eac8-0778-5f1a-0f28b8c2a227|female|40              |4.0                |15                    |0               |
|0afc0e1f-71ef-2925-ef91-adc6333145ba|male  |39              |5.0                |13                    |0               |
|72b5373b-e9f1-db2d-30db-67fa208b7359|male  |25              |1.0                |7                     |0               |
|47697c8b-11d6-e042-4e39-c3b590e548b7|female|74              |1.0                |29                    |0               |
+---------------

In [0]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Encode gender as numeric
gender_indexer = StringIndexer(inputCol="gender", outputCol="gender_index")

# Assemble features into a vector
assembler = VectorAssembler(
    inputCols=["gender_index", "age_at_admission", "length_of_stay_days", "active_condition_count"],
    outputCol="features"
)

# Random Forest
rf = RandomForestClassifier(
    labelCol="readmitted_30day",
    featuresCol="features",
    numTrees=100,
    seed=42
)

# Build pipeline
pipeline = Pipeline(stages=[gender_indexer, assembler, rf])

# Split data
train, test = encounters_featured.randomSplit([0.8, 0.2], seed=42)

# Train model
model = pipeline.fit(train)

# Evaluate
predictions = model.transform(test)
evaluator = BinaryClassificationEvaluator(labelCol="readmitted_30day")
auc = evaluator.evaluate(predictions)
print(f"AUC: {auc:.3f}")

AUC: 0.000


In [0]:
print("Train readmissions:", train.filter(col("readmitted_30day") == 1).count())
print("Test readmissions:", test.filter(col("readmitted_30day") == 1).count())

Train readmissions: 4
Test readmissions: 0


In [0]:
import os
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/databricks_fhir_r4_synthetic_data/synthea_fhir_ext/synthea_fhir_1k_sample_r4"

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.google_fhir_data.ml_temp")

DataFrame[]

In [0]:
import os
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/google_fhir_data/ml_temp"

In [0]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(labelCol="readmitted_30day")

# Empty param grid — just doing cross-validation, not tuning yet
paramGrid = ParamGridBuilder().build()

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=4,  # 4 folds so each fold has at least 1 readmission
    seed=42
)

cv_model = cv.fit(encounters_featured)

print(f"Average AUC across folds: {max(cv_model.avgMetrics):.3f}")

Average AUC across folds: 0.277


In [0]:
encounters_featured.write.format("delta").mode("overwrite").saveAsTable("workspace.google_fhir_data.encounters_featured")

patients_clean.write.format("delta").mode("overwrite").saveAsTable("workspace.google_fhir_data.patients_clean")

In [0]:
readmissions.write.format("delta").mode("overwrite").saveAsTable("workspace.google_fhir_data.readmissions")